# ⚠️⚠️⚠️ A NE PAS EXECUTER CE NOTENOOK ⚠️⚠️⚠️
# Analyse et Entraînement d'un Modèle de Classification d'Émotions

Ce notebook présente le **processus complet d'entraînement d'un modèle de classification d'émotions** à partir d'un jeu de données textuelles. Voici les points principaux :

- **Environnement** : Le script est exécuté sur **Kaggle** afin de bénéficier du **GPU gratuit**, ce qui accélère considérablement l'entraînement d'un modèle lourd.

- **Modèle de base** : Nous utilisons **`roberta-base`**, un modèle pré-entraîné de type Transformer.  
  - Bien que relativement lourd, ce modèle est choisi pour sa **capacité à capturer des nuances linguistiques complexes** et à obtenir de bonnes performances sur les tâches de classification textuelle.

- **Jeu de données** :
  - Composé de **2 colonnes** :
    1. `text` : le texte en anglais.
    2. `label` : l'étiquette correspondant à l'émotion, parmi **7 catégories possibles**.
  - L'objectif est de prédire correctement l'émotion associée à chaque texte.

- **Objectif du notebook** : 
  - Montrer le **flux complet** depuis le prétraitement des données, la tokenisation, l'entraînement du modèle, jusqu'à l'évaluation des performances.
  - Fournir un exemple pratique d'utilisation d'un modèle Transformer sur un problème de classification multi-classes.

> Ce notebook est conçu pour être **didactique** tout en permettant de reproduire l'entraînement d'un modèle robuste sur des données réelles.

In [1]:
# =============================
# 🌐 UTILISER LES MODÈLES DEPUIS LES DATASETS KAGGLE
# =============================
print("📥 Configuration pour utiliser les modèles Kaggle...")

import os
import subprocess
import sys


📥 Configuration pour utiliser les modèles Kaggle...


In [2]:
# =============================
# ÉTAPE 1: Installation des packages
# =============================
print("\n📦 Installation des packages...")
packages = ["transformers", "datasets", "scikit-learn", "tqdm", "polars", "torch", "tokenizers"]

for pkg in packages:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        print(f"   ✅ {pkg}")
    except:
        print(f"   ⚠️  {pkg} (déjà installé)")



📦 Installation des packages...
   ✅ transformers
   ✅ datasets
   ✅ scikit-learn
   ✅ tqdm
   ✅ polars
   ✅ torch
   ✅ tokenizers


Le CSV est composé de **trois datasets** récupérés depuis **HuggingFace**, combinés afin d'obtenir les **7 labels** que j'ai préparés (voir le script `prep_dataset_en_7_labels.py`).  
Le dataset est ensuite **importé dans Kaggle** afin d'être utilisé pour l'entraînement.


In [ ]:
# =============================
# 2 Charger le CSV depuis les fichiers d'input Kaggle
# =============================
import polars as pl

input_path = "/kaggle/input/"

csv_files = []
for root, dirs, files in os.walk(input_path):
    for file in files:
        if file.endswith('.csv'):
            csv_files.append(os.path.join(root, file))

if not csv_files:
    raise FileNotFoundError(f"❌ Aucun fichier CSV trouvé dans {input_path}")

filename = csv_files[0]
print(f"📂 Fichier utilisé: {filename}")

df_full = pl.read_csv(filename)

print(f"\n📊 Dataset: {len(df_full):,} lignes")
print(f"📋 Colonnes: {df_full.columns}")

required_cols = ["text_clean", "label", "label_id"]
missing = [c for c in required_cols if c not in df_full.columns]
if missing:
    raise ValueError(f"❌ Colonnes manquantes: {missing}")

print("\n📊 Distribution labels:")
print(df_full.group_by("label").len().sort("label"))


📂 CHARGEMENT DU DATASET
📂 Fichier utilisé: /kaggle/input/train-test-dataset/test.csv

📊 Dataset: 42,000 lignes
📋 Colonnes: ['text_clean', 'label', 'label_id']

📊 Distribution labels:
shape: (7, 2)
┌──────────┬──────┐
│ label    ┆ len  │
│ ---      ┆ ---  │
│ str      ┆ u32  │
╞══════════╪══════╡
│ anger    ┆ 6000 │
│ fear     ┆ 6000 │
│ joy      ┆ 6000 │
│ love     ┆ 6000 │
│ neutral  ┆ 6000 │
│ sad      ┆ 6000 │
│ surprise ┆ 6000 │
└──────────┴──────┘


In [ ]:
# =============================
# 3️ Convertir en pandas + HF Dataset
# =============================
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split

df = df_full.to_pandas()

train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df['label_id'],
    random_state=42
)
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

print(f"\n📊 Train set: {len(train_df):,} samples")
print(f"📊 Val set: {len(val_df):,} samples")



🔄 PRÉPARATION DES DONNÉES

📊 Train set: 37,800 samples
📊 Val set: 4,200 samples


Pour améliorer l'entraînement du modèle, on crée des **versions raccourcies des textes** du jeu d'entraînement afin que le model entraîner puisse predire les phrases de différentes longueurs. Les textes raccourcis sont **ajoutés au jeu d'entraînement**, doublant ainsi le nombre d'exemples et enrichissant les données disponibles pour l'apprentissage.  

En résumé : cette étape **augmente le dataset** de façon simple pour que le modèle voie plus de variations de texte et **mieux reconnaisse les émotions**, même sur des phrases plus courtes.

 

In [ ]:
# =============================
# 4️ Data augmentation
# =============================

import random


def create_short_text(text, min_words=3, max_words=15):
    words = str(text).split()
    if len(words) <= max_words:
        return text
    start = random.randint(0, max(0, len(words) - max_words))
    end = start + random.randint(min_words, min(max_words, len(words) - start))
    return " ".join(words[start:end])

short_texts = []
for i, row in train_df.iterrows():
    short_version = create_short_text(row['text_clean'])
    short_texts.append({
        "text_clean": short_version, 
        "label": row['label'], 
        "label_id": row['label_id']
    })

train_df_aug = pd.concat([train_df, pd.DataFrame(short_texts)], ignore_index=True)
train_dataset = Dataset.from_pandas(train_df_aug)

print(f"✅ Dataset augmenté: {len(train_dataset):,} samples")


🔀 Data augmentation en cours...
✅ Dataset augmenté: 75,600 samples


### Tokenisation des textes avec `roberta-base`
Avant d'entraîner un modèle Transformer, il faut **convertir les textes en tokens** que le modèle peut comprendre.  

1. **Chargement du tokenizer** :  
   - `AutoTokenizer.from_pretrained(MODEL_NAME)` charge le tokenizer associé à `roberta-base`.  
   - Le tokenizer transforme le texte en **séquence de nombres**, correspondant aux tokens connus du modèle.

2. **Définition de la longueur maximale** :  
   - `MAX_LEN = 128` : on limite chaque texte à 128 tokens.  
   - Les textes plus courts sont **complétés par du padding**, les textes plus longs sont **tronqués**.

3. **Fonction de tokenisation** :  
   - `tokenize_function` prend une série de textes (`text_clean`) et les convertit en tokens avec padding et troncature.  
   - Cela permet au modèle de recevoir des entrées de **longueur fixe**, indispensable pour l'entraînement.

4. **Application sur les datasets** :  
   - `train_dataset.map(tokenize_function, batched=True)` applique la tokenisation à **tous les textes du jeu d’entraînement**.  
   - Même chose pour le jeu de validation (`val_dataset`).  
   - Le paramètre `batched=True` permet de traiter plusieurs exemples en même temps pour gagner du temps.

In [6]:
from transformers import AutoTokenizer

MODEL_NAME = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LEN = 128  # court texte

def tokenize_function(examples):
    return tokenizer(
        examples["text_clean"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/75600 [00:00<?, ? examples/s]

Map:   0%|          | 0/4200 [00:00<?, ? examples/s]

# Création d'un Dataset PyTorch pour l'entraînement

Pour entraîner un modèle avec PyTorch, il faut convertir les **datasets HuggingFace** en objets compatibles avec PyTorch.  

1. **Définition de la classe `EmotionDataset`** :  
   - Hérite de `torch.utils.data.Dataset`, ce qui permet de l'utiliser directement avec un **DataLoader PyTorch**.  
   - La classe prend en entrée un **dataset HuggingFace** (`hf_dataset`).  

2. **Méthodes principales** :  
   - `__len__` : retourne le nombre total d'exemples dans le dataset.  
   - `__getitem__` : retourne un exemple à l'indice `idx`, avec les clés suivantes :  
     - `input_ids` : la séquence de tokens du texte.  
     - `attention_mask` : masque indiquant quelles positions du texte sont pertinentes (utile pour gérer le padding).  
     - `labels` : l’étiquette de l’exemple (la classe d’émotion).  
   - Tous ces éléments sont convertis en **tensors PyTorch** (`torch.tensor`) pour être utilisés directement par le modèle.

3. **Création des datasets PyTorch** :  
   - `train_data = EmotionDataset(train_dataset)`  
   - `val_data = EmotionDataset(val_dataset)`  
   - Ces objets peuvent maintenant être utilisés avec un **DataLoader** pour l’entraînement et la validation.


In [ ]:
import torch


class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset):
        self.dataset = hf_dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(item["label_id"], dtype=torch.long),
        }

train_data = EmotionDataset(train_dataset)
val_data = EmotionDataset(val_dataset)


In [9]:
from transformers import AutoModelForSequenceClassification

NUM_LABELS = 7  # anger, fear, joy, love, neutral, sad, surprise

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)


2026-02-08 15:05:00.818765: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770563101.027435      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770563101.087533      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770563101.603505      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770563101.603551      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770563101.603554      55 computation_placer.cc:177] computation placer alr

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Définition des arguments d'entraînement

Pour entraîner le modèle Transformer, on utilise `TrainingArguments` de HuggingFace, qui configure tous les paramètres d'apprentissage. Voici pourquoi les choix faits sont adaptés à notre dataset :  

1. **Répertoire de sortie** :  
   - `output_dir="./results"` stocke les résultats et checkpoints du modèle.

2. **Nombre d'époques** :  
   - `num_train_epochs=3` : avec un dataset de taille modérée, 3 époques suffisent pour que le modèle apprenne sans surapprentissage.

3. **Taille de batch** :  
   - `per_device_train_batch_size=16` et `per_device_eval_batch_size=16` : taille raisonnable pour le GPU gratuit de Kaggle et adaptée à la taille du dataset.  
   - Permet un bon compromis entre vitesse et stabilité de l'entraînement.

4. **Taux d'apprentissage** :  
   - `learning_rate=2e-5` : une valeur standard pour les modèles pré-entraînés comme `roberta-base`, efficace pour la classification de texte.

5. **Logging et reporting** :  
   - `logging_steps=50` affiche les progrès régulièrement sans surcharger la console.  
   - `report_to="none"` désactive l'intégration avec des outils externes (ex: WandB) pour simplifier l’exécution sur Kaggle.

6. **Précision mixte** :  
   - `fp16=True` utilise le **half-precision floating point**, ce qui accélère l’entraînement et réduit la mémoire utilisée sur GPU.

7. **Sauvegarde** :  
   - `save_strategy="no"` : on ne sauvegarde pas de checkpoints intermédiaires, suffisant pour des tests rapides et pour ne pas saturer l’espace de stockage sur Kaggle.

✅ **En résumé** : ces paramètres sont adaptés à notre dataset de taille modérée et au GPU gratuit de Kaggle, tout en permettant un entraînement efficace et stable.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    logging_steps=50,
    do_train=True,
    do_eval=True,
    report_to="none",
    fp16=True,
    save_strategy="no" 
)


In [ ]:
from sklearn.metrics import accuracy_score, f1_score


# Fonction de calcul des métriques
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": f1}

In [18]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    compute_metrics=compute_metrics
)

trainer.train()


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
50,0.379900
100,0.395900
150,0.386100
200,0.362500
250,0.362700
300,0.330600
350,0.349700
400,0.341300
450,0.381200
500,0.382200


TrainOutput(global_step=7089, training_loss=0.27993428392661845, metrics={'train_runtime': 3241.4326, 'train_samples_per_second': 69.969, 'train_steps_per_second': 2.187, 'total_flos': 1.491906657024e+16, 'train_loss': 0.27993428392661845, 'epoch': 3.0})

In [ ]:
test_texts = [
    "I am happy that I did it",
    "I love spending time with my family",
    "I hate when people lie to me",
    "It's just another normal day",
    "She is sad because she lost her favorite book",
    "I feel anxious before every presentation",
    "I can't stop thinking about that betrayal",
    "He is proud of his accomplishments",
    "The sunset over the mountains fills me with awe and love for nature",
    "After months of hard work, seeing the results brought tears of joy and relief",
    "The constant noise and chaos in the city makes me stressed and frustrated",
    "Losing someone you care about leaves a hollow feeling of sadness",
    "Watching the baby take its first steps filled me with happiness",
    "She was terrified when she realized she was lost in the dark forest",
    "He felt a mix of hate and betrayal after being lied to by his closest friend"
    ]

In [21]:
# tokenize les textes
# Tokenization
encoded_inputs = tokenizer(
    test_texts,
    truncation=True,
    padding="max_length",
    max_length=128,
    return_tensors="pt"  # PyTorch tensors
)

# Déplacer sur GPU si disponible
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
input_ids = encoded_inputs["input_ids"].to(device)
attention_mask = encoded_inputs["attention_mask"].to(device)

In [22]:
model.eval()  # mode évaluation
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits
    preds = torch.argmax(logits, dim=-1)  # indices des labels prédits


In [29]:
id2label = {i: label for i, label in enumerate(df["label"].unique())}

pred_labels = [id2label[int(p)] for p in preds]

for text, label in zip(test_texts, pred_labels):
    print(f"Texte: {text}\nPredicted label: {label}\n")


Texte: I am happy that I did it
Predicted label: joy

Texte: I love spending time with my family
Predicted label: joy

Texte: I hate when people lie to me
Predicted label: anger

Texte: It's just another normal day
Predicted label: neutral

Texte: She is sad because she lost her favorite book
Predicted label: sad

Texte: I feel anxious before every presentation
Predicted label: fear

Texte: I can't stop thinking about that betrayal
Predicted label: sad

Texte: He is proud of his accomplishments
Predicted label: joy

Texte: The sunset over the mountains fills me with awe and love for nature
Predicted label: surprise

Texte: After months of hard work, seeing the results brought tears of joy and relief
Predicted label: surprise

Texte: The constant noise and chaos in the city makes me stressed and frustrated
Predicted label: anger

Texte: Losing someone you care about leaves a hollow feeling of sadness
Predicted label: sad

Texte: Watching the baby take its first steps filled me with happ

In [28]:
# Chemin de sauvegarde
save_dir = "/kaggle/working/roberta_final"

# Sauvegarde du modèle et du tokenizer
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print("✅ Modèle et tokenizer sauvegardés avec save_pretrained")



✅ Modèle et tokenizer sauvegardés avec save_pretrained


# Remarque sur le modèle entraîné

Le modèle de base **`roberta-base`** est déjà un modèle assez lourd.  
Après entraînement, le modèle devient **encore plus volumineux** et n'est donc pas facile à partager directement sur GitHub.  

Pour contourner ce problème, le modèle a été **uploadé sur HuggingFace**.  
Il peut ainsi être **récupéré et utilisé facilement** depuis n'importe quel notebook ou script.  

> Un exemple d'utilisation du modèle depuis HuggingFace est disponible dans `test_hugging_face_model.ipynb`.


sauvegarde en zip si besoin

In [ ]:
import shutil

save_dir = "/kaggle/working/roberta_final"
zip_path = "/kaggle/working/roberta_final.zip"

# Créer le zip
shutil.make_archive(base_name=zip_path.replace(".zip",""), format='zip', root_dir=save_dir)

print(f"✅ Modèle compressé dans : {zip_path}")
